# Schema design — FARS 2024
**Saida** · DATA 201 group project

Goal: decide the entities, keys, relationships and EER constructs for the ER diagram,
and prove every decision against the cleaned data in `data/processed/`.

Rule followed throughout: **nothing is assumed because a name sounds right.**
Each step prints the evidence for one diagram decision.


## Step 0 — setup

In [4]:
import csv, collections
from pathlib import Path

D = Path("../data/processed")
files = sorted(p.name for p in D.glob("*.csv"))
print(len(files), "CSV files")
print(files)

0 CSV files
[]


## Step 1 — how big is each table?

Large tables are usually entities; small ones are usually child/attribute tables.

In [5]:
sizes = {}
for p in D.glob("*.csv"):
    if p.name == "vpicdecode.csv":      # 44 MB, skip for speed
        continue
    with open(p) as f:
        sizes[p.name] = sum(1 for _ in f) - 1   # minus header

for n, r in sorted(sizes.items(), key=lambda x: -x[1]):
    print(f"{r:>8,}  {n}")

## Step 2 — what columns does each table have?

The key columns tell you the **grain**: one row per what?

In [6]:
for n in ["accident.csv", "vehicle.csv", "person.csv", "parkwork.csv", "drugs.csv"]:
    cols = next(csv.reader(open(D / n)))
    print(f"== {n}  ({len(cols)} cols)")
    print("   ", cols[:12], "...\n")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/accident.csv'

## Step 3 — group all tables by grain

A table belongs to the crash / vehicle / person level depending on which key
columns it carries. **This is the step that turns 36 tables into 3 entities.**

In [ ]:
grain = collections.defaultdict(list)
for p in sorted(D.glob("*.csv")):
    cols = set(next(csv.reader(open(p))))
    if   {"PER_NO"} <= cols: grain["person  (ST_CASE,VEH_NO,PER_NO)"].append(p.name)
    elif {"VEH_NO"} <= cols: grain["vehicle (ST_CASE,VEH_NO)"].append(p.name)
    elif {"ST_CASE"} <= cols: grain["crash   (ST_CASE)"].append(p.name)
    else: grain["other"].append(p.name)

for g, fs in grain.items():
    print(f"{g}  -> {len(fs)} tables")
    print("   ", ", ".join(fs), "\n")

### What this means for the diagram
Only the **grains** become entities. The other tables are repeating attribute groups
hanging off a grain — drawing 36 boxes would be noise, not detail.

## Step 4 — is the candidate key actually unique?

Zero duplicates = the key holds. Non-zero = you need another column or a surrogate.

In [ ]:
def uniq(fname, keys):
    rows = list(csv.DictReader(open(D / fname)))
    s = {tuple(r[k] for k in keys) for r in rows}
    print(f"{fname:<16} {keys}  rows={len(rows):,}  distinct={len(s):,}  dupes={len(rows)-len(s):,}")
    return rows

acc = uniq("accident.csv", ["ST_CASE"])
veh = uniq("vehicle.csv",  ["ST_CASE","VEH_NO"])
per = uniq("person.csv",   ["ST_CASE","VEH_NO","PER_NO"])
pwk = uniq("parkwork.csv", ["ST_CASE","VEH_NO"])
_   = uniq("drugs.csv",    ["ST_CASE","VEH_NO","PER_NO"])   # expect duplicates

## Step 5 — strong entity or weak entity?

A key column that **restarts inside its parent** is a *partial key*, which makes the
entity weak. Test it: how many distinct values does the column have on its own?

In [ ]:
for name, rows, col in [("vehicle", veh, "VEH_NO"), ("person", per, "PER_NO")]:
    d = len({r[col] for r in rows})
    print(f"{name}.{col}: {d} distinct values across {len(rows):,} rows "
          f"-> {'partial key (WEAK entity)' if d < 100 else 'possibly unique'}")

## Step 6 — does every join actually work?

A relationship only exists if the rows match. Count the orphans **before** drawing it.

In [ ]:
veh_keys = {(r["ST_CASE"], r["VEH_NO"]) for r in veh}
pwk_keys = {(r["ST_CASE"], r["VEH_NO"]) for r in pwk}

orphans = [r for r in per if r["VEH_NO"] != "0" and (r["ST_CASE"], r["VEH_NO"]) not in veh_keys]
print("person rows with VEH_NO>0 that match NO vehicle:", len(orphans))
print("  ...of those, matching a parkwork vehicle:",
      sum(1 for r in orphans if (r["ST_CASE"], r["VEH_NO"]) in pwk_keys))
print("parkwork rows that match a vehicle:", len(pwk_keys & veh_keys), "of", len(pwk_keys))

### Why this step matters
If orphans exist, the foreign key you were about to draw **would fail on load**.
Here it exposes that `PERSON.VEH_NO` does not always point at `vehicle`.

## Step 7 — participation: total or partial?

Count parents with **zero** children. `0` -> total (double line). `>0` -> partial (single line).

In [ ]:
acc_ids  = {r["ST_CASE"] for r in acc}
veh_cases= {k[0] for k in veh_keys}
per_cases= {r["ST_CASE"] for r in per}
occupied = {(r["ST_CASE"], r["VEH_NO"]) for r in per if r["VEH_NO"] != "0"}

print("crashes with 0 vehicles :", len(acc_ids - veh_cases), " -> ACCIDENT side of IN_CRASH")
print("crashes with 0 persons  :", len(acc_ids - per_cases), " -> ACCIDENT side of INCLUDES")
print("vehicles with 0 occupants:", len(veh_keys - occupied), " -> VEHICLE side of OCCUPIES")

## Step 8 — is an EER specialization justified?

Look for a column that changes **which table a row relates to**. Cross-tabulate it
against where the foreign key actually lands.

In [ ]:
ct = collections.Counter()
for r in per:
    k = (r["ST_CASE"], r["VEH_NO"])
    where = ("VEH_NO=0" if r["VEH_NO"] == "0"
             else "vehicle" if k in veh_keys
             else "parkwork" if k in pwk_keys else "ORPHAN")
    ct[(r["PER_TYP"], where)] += 1

for (t, w), n in sorted(ct.items(), key=lambda x: -x[1]):
    print(f"  PER_TYP={t:>2}  {w:<9} {n:>7,}")

print("\nPER_TYP labels:")
for r in csv.DictReader(open(D / "lookups" / "code_labels.csv")):
    if r["column"] == "PER_TYP":
        print("   ", r["code"], "=", r["label"])

### Reading the result
Each `PER_TYP` value maps to exactly one destination, with no ORPHAN rows.
That is a **disjoint, total, attribute-defined specialization** — the subtypes are
real, not decorative. If no column had split cleanly, the honest answer would be
"plain ER, no EER extension needed".

## Step 9 — confirm the subtype-only child tables

Tables that belong to one subtype only are extra evidence for the specialization.

In [ ]:
nm = {(r["ST_CASE"], r["VEH_NO"], r["PER_NO"]) for r in per if r["VEH_NO"] == "0"}
for t in ["pbtype", "safetyeq", "nmcrash", "nmdistract", "nmimpair", "nmprior"]:
    s = {(r["ST_CASE"], r["VEH_NO"], r["PER_NO"]) for r in csv.DictReader(open(D / f"{t}.csv"))}
    print(f"{t:<12} {len(s):>6,} persons, outside the non-motorist subtype: {len(s - nm)}")

## Summary — what the diagram should show

Fill this in from the outputs above:

| Decision | Evidence | Step |
|---|---|---|
| ACCIDENT is a strong entity | `ST_CASE` unique, 0 dupes | 4 |
| VEHICLE / PERSON are weak | `VEH_NO` / `PER_NO` restart per parent | 5 |
| PARKED_WORK_VEH is separate, no FK to VEHICLE | 0 of 1,526 rows join | 6 |
| ACCIDENT total in both relationships | 0 crashes with no vehicle / no person | 7 |
| VEHICLE partial in OCCUPIES | 292 vehicles with no occupant | 7 |
| PERSON specializes 3 ways on `PER_TYP` | clean split, 0 orphans | 8 |
| DRUG_TEST needs a surrogate key | duplicate rows | 4 |

Anything the data could **not** settle goes in an assumptions list under the diagram.
